# Execution and checkpoint publication
Local environment: from this directory use `uv sync`, then select `.venv/bin/python` as the notebook kernel. The recorded execution used the existing ADA Python 3.12.4 / PyTorch 2.6.0 CUDA 12.4 environment; actual package versions appear in the setup output. The laptop has no working CUDA driver at preparation time. Downloaded data and checkpoints can be used locally for inspection and CPU inference.

The scheduler scripts below are the exact launch configuration used for preparation and the five full runs. They use a new ADA directory, `~/anlp_a2/part1_20260922`, and preserve the previous Model 1 experiment. Use `squeue -u $USER` and job logs on ADA to monitor. Executed notebooks are saved after every completed cell and long-training checkpoints after every validation interval. A cell interruption may leave its notebook output incomplete; the binary resume checkpoint retains optimizer and RNG state.

The PDF asks for model checkpoints on Hugging Face. The upload cell is disabled until a repository namespace is provided and publication is explicitly enabled. It uploads each trained checkpoint with its tokenizer and exact implementation; it never uploads credentials or the dataset.


In [ ]:
from pathlib import Path
import json, os
here=Path.cwd().resolve()
ROOT=next((p for p in [here,*here.parents,here/'part 1'] if (p/'shared'/'implementation.ipynb').exists()),None)
assert ROOT is not None, 'Set the notebook working directory inside part 1.'
for cell in json.loads((ROOT/'shared'/'implementation.ipynb').read_text())['cells']:
    if cell['cell_type']=='code' and 'definitions' in cell.get('metadata',{}).get('tags',[]):
        exec(compile(cell['source'],'shared/implementation.ipynb','exec'),globals())


In [ ]:
# Scheduler IDs returned by ADA when these experiments were submitted.
ADA_PROJECT='~/anlp_a2/part1_20260922'
PREPARATION_JOB=2711534
TRAINING_ARRAY=2711535
print({'ADA project':ADA_PROJECT,'Preparation job':PREPARATION_JOB,'Training array':TRAINING_ARRAY})
# On ADA: sacct -j 2711534,2711535 --format=JobID,JobName,State,Elapsed,ExitCode


In [ ]:
PREPARE_SBATCH='#!/bin/bash\n#SBATCH --job-name=a2-p1-prepare\n#SBATCH --account=research\n#SBATCH --partition=u22\n#SBATCH --qos=medium\n#SBATCH --constraint=2080ti\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=20G\n#SBATCH --time=01:00:00\n#SBATCH --output=/home2/divyansh.atri/anlp_a2/part1_20260922/prepare-%j.log\nset -euo pipefail\nsource "$HOME/anlp_a2/slurm/env.sh"\ncd "$HOME/anlp_a2/part1_20260922"\nexport HF_HOME="$HOME/anlp_a2/.cache/huggingface"\nexport MPLCONFIGDIR="$PWD/.cache/matplotlib"\nexport PART1_PREPARE=1\nexport TOKENIZERS_PARALLELISM=false\nexport OMP_NUM_THREADS=4\nmkdir -p .runs\npapermill 00_setup_and_checks.ipynb .runs/setup.executed.ipynb --cwd "$PWD" --log-output --request-save-on-cell-execute\ncp .runs/setup.executed.ipynb 00_setup_and_checks.ipynb\n'
MODELS_SBATCH='#!/bin/bash\n#SBATCH --job-name=a2-p1-model\n#SBATCH --account=research\n#SBATCH --partition=u22\n#SBATCH --qos=medium\n#SBATCH --constraint=2080ti\n#SBATCH --gres=gpu:1\n#SBATCH --cpus-per-task=4\n#SBATCH --mem=16G\n#SBATCH --time=04:00:00\n#SBATCH --array=1-5%5\n#SBATCH --output=/home2/divyansh.atri/anlp_a2/part1_20260922/model-%A-%a.log\nset -euo pipefail\nsource "$HOME/anlp_a2/slurm/env.sh"\ncd "$HOME/anlp_a2/part1_20260922"\nexport HF_HOME="$HOME/anlp_a2/.cache/huggingface"\nexport MPLCONFIGDIR="$PWD/.cache/matplotlib-$SLURM_ARRAY_TASK_ID"\nexport PART1_PROFILE=full\nexport TOKENIZERS_PARALLELISM=false\nexport OMP_NUM_THREADS=4\nMODEL="$SLURM_ARRAY_TASK_ID"\nINPUT="model $MODEL/model_$MODEL.ipynb"\nmkdir -p .runs\nOUTPUT=".runs/model_$MODEL.executed.ipynb"\nnvidia-smi --query-gpu=name,memory.total --format=csv,noheader\npapermill "$INPUT" "$OUTPUT" --cwd "$PWD" --log-output --request-save-on-cell-execute\ncp "$OUTPUT" "$INPUT"\n'
print(PREPARE_SBATCH)
print(MODELS_SBATCH)


In [ ]:
# Execute on ADA after copying this directory; submission is opt-in.
SUBMIT_JOBS=False
if SUBMIT_JOBS:
    import subprocess
    launch=ROOT/'.runs'; launch.mkdir(exist_ok=True)
    prep=launch/'prepare.sbatch'; prep.write_text(PREPARE_SBATCH)
    jobs=launch/'models.sbatch'; jobs.write_text(MODELS_SBATCH)
    preparation=subprocess.check_output(['sbatch','--parsable',str(prep)],text=True).strip()
    training=subprocess.check_output(['sbatch','--parsable',f'--dependency=afterok:{preparation}',str(jobs)],text=True).strip()
    print({'preparation_job':preparation,'training_array':training})


In [ ]:
UPLOAD_CHECKPOINTS=False
HF_NAMESPACE='' # e.g. your Hugging Face username or organization; never paste a token here.
if UPLOAD_CHECKPOINTS:
    assert HF_NAMESPACE and '/' not in HF_NAMESPACE
    from huggingface_hub import HfApi
    api=HfApi(); identity=api.whoami(); links=[]
    for i in range(1,6):
        checkpoint=ROOT/f'model {i}'/'full'/'checkpoint.pt'
        assert checkpoint.exists(),f'Model {i} is not trained.'
        repo=f'{HF_NAMESPACE}/anlp-assignment2-part1-model{i}'
        api.create_repo(repo_id=repo,repo_type='model',private=False,exist_ok=True)
        for path,name in [(checkpoint,'checkpoint.pt'),(ROOT/'shared'/'tokenizer_clean.json','tokenizer.json'),(ROOT/'shared'/'implementation.ipynb','implementation.ipynb')]:
            api.upload_file(path_or_fileobj=str(path),path_in_repo=name,repo_id=repo,repo_type='model',commit_message=f'Upload Part 1 Model {i} artifact')
        links.append({'model':i,'url':f'https://huggingface.co/{repo}','checkpoint_sha256':sha(checkpoint)})
    display(pd.DataFrame(links))
else:
    print('Hugging Face publication has not been requested in this notebook session.')
